In [ ]:
import pandas as pd
import joblib
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

MODEL_DIR = "all-roberta-large-v1"
GPT5_INPUT_PRICE  = 1.25  / 1_000_000
GPT5_OUTPUT_PRICE = 10.0  / 1_000_000
PERCENTILE = 0.95

# ── Load data ──────────────────────────────────────────────────────────────────
train_df = pd.read_csv("dataset/final_binary/train.csv")
test_df  = pd.read_csv("dataset/final_binary/test.csv")

# ── Define high-cost label using train-set percentile ─────────────────────────
percentile_bound = train_df["output_tokens_gpt5"].quantile(PERCENTILE)
train_df["high_cost"] = (train_df["output_tokens_gpt5"] > percentile_bound).astype(int)
test_df["high_cost"]  = (test_df["output_tokens_gpt5"]  > percentile_bound).astype(int)

avg_input = train_df["input_tokens_gpt5"].mean()
print(f"{PERCENTILE*100:.0f}th percentile of GPT-5 output tokens : {percentile_bound:.0f} tokens")
print(f"Estimated cost at threshold                   : "
      f"{percentile_bound * GPT5_OUTPUT_PRICE + avg_input * GPT5_INPUT_PRICE:.6f} USD/query")
print(f"High-cost fraction (train) : {train_df['high_cost'].mean():.2%}")
print(f"High-cost fraction (test)  : {test_df['high_cost'].mean():.2%}\n")

# ── Encode questions ───────────────────────────────────────────────────────────
print("Loading embedder...")
encoder = SentenceTransformer(MODEL_DIR)

print("Encoding train questions...")
X_train = encoder.encode(train_df["question"].tolist(), normalize_embeddings=True, show_progress_bar=True)

print("Encoding test questions...")
X_test  = encoder.encode(test_df["question"].tolist(),  normalize_embeddings=True, show_progress_bar=True)

y_train = train_df["high_cost"].values
y_test  = test_df["high_cost"].values

In [ ]:
# ── Build ensemble classifier (class_weight='balanced' handles imbalance) ──────
estimators = [
    ("lr",  LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)),
    # ("gb", GradientBoostingClassifier(n_estimators=200, random_state=42)),
    ("svc", SVC(kernel="rbf", C=1.0, class_weight="balanced", probability=True, random_state=42)),
    ("mlp", MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)),
]

clf = VotingClassifier(estimators=estimators, voting="soft")

print("Training classifier...")
clf.fit(X_train, y_train)

def print_stats(name, y_true, y_pred):
    print(f"\n── {name} ──────────────────────────────────────────────────")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"F1        : {f1_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, target_names=["low cost (0)", "high cost (1)"]))

# ── Individual model stats ─────────────────────────────────────────────────────
for name, estimator in clf.named_estimators_.items():
    print_stats(name.upper(), y_test, estimator.predict(X_test))

# ── Ensemble stats ─────────────────────────────────────────────────────────────
y_pred = clf.predict(X_test)
print_stats("Ensemble (VotingClassifier)", y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["low cost (0)", "high cost (1)"])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — High-cost classifier")
plt.tight_layout()
plt.show()

# ── Save model ─────────────────────────────────────────────────────────────────
os.makedirs("trained_routers", exist_ok=True)
joblib.dump(clf, "trained_routers/high_cost_classifier.joblib")
print("\nModel saved to trained_routers/high_cost_classifier.joblib")
